# Drexel University
## College of Computing and Informatics
## INFO 371: Data Mining Applications
## Assignment 3


## **DON'T FORGET TO PUT YOUR NAME BELOW**

## **NAME:**


### A. What to Hand In

1. A completed this Jupyter notebook.
2. Any data sets used in the assignment.
3. Any multimedia and images used in the assignment.

### B. How to Hand In

Submit your files through the course website in the Blackboard Learn system.

### C. When to Hand In

1. Submit your assignment no later than 11:59pm in the due date.
2. There will be a 10% (absolute value) deduction for each day of lateness, to a maximum of 3 days; assignments will not be accepted beyond that point. Missing work will earn a zero grade.

### D. Answer the following questions

# Import packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib


# Import the classifiers: KNeighborsClassifier, DecisionTreeClassifier, RandomForestClassifier, and MultinomialNB

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

# Import CountVectorizer and TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Import evaluation metrics

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Import the train_test_split function

In [ ]:
from sklearn.model_selection import train_test_split

# Answer the following questions

## Question 1 [5 points]: Collecting and uploading the movie review data.
In this assignment, you will apply various classification models and text vectorizations for sentiment analysis on movie reviews. The original data are from: https://www.kaggle.com/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews It contains 50K movie reviews. To reduce the size of the data, I have extracted 5K random reviews to this file: `imdb_reviews_5k.csv`. You can work on this small data set in this assignment. Now, upload the 5k reviews and read them in as a DataFrame. Show the columns and size of the data. Display the first 5 rows.

In [ ]:
imdb_file = 'data/imdb_reviews_5k.csv'
df = pd.read_csv(imdb_file)
#print colums and rows and first 3 entires
print(df.columns)
print(df.shape)
df.head()

## Question 2 [5 points]: Show the distribution of the sentiment values (positive and negative). Are the values evenly distributed in the data set?

In [ ]:
#count sentiment values
sentiment_count = df['sentiment'].value_counts()
#print counts
print('sentiment distribution')
print(sentiment_count)

#plot sentiment values
plt.figure()
sentiment_count.plot(kind = 'bar')
plt.xlabel('sentiment')
plt.ylabel('count')
plt.title('sentiment distribution')
plt.show()

#get percentage distribution
percentage = df['sentiment'].value_counts(normalize = True) * 100
print('percentage distribution')
print(percentage)

# are the values evenly distributied
#yes the values are distributed , meaning dataset is very close to being evenly balanced


## Question 3 [10 points]: You will use the sentiment values as classification labels. Extract the target labels `y` by converting the sentiment values to 1 and 0.  

In [ ]:
# convert sentiment labels to binary values
df['y'] = df['sentiment'].map({'positive': 1, 'negative': 0})

#extract target label
y = df['y']

#print results
print(y.head())
print("Value Counts: ", y.value_counts())

## Question 4 [10 points]: Prepare training and test data sets.
Split the review text and target labels into review_train, review_test, y_train, and y_test. Use 80% of the data for taining and 20% for test. Do you need to consider statification when splitting?

In [ ]:
X = df['review']
y = df['y']

#split into 80 train, 20 test

review_train, review_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # ensure balnace distribution
)

#chesk sizes
print('Training set size:', len(review_train))
print('Test set size:', len(review_test))

print('\n Training label dsitribution')
print(y_train.value_counts())

print('\n Test label dsitribution')
print(y_test.value_counts())

#do you need stratitifcaiton
# yes you need stratification , it ensures The train and test sets maintain the same positive/negative ratio
#No accidental imbalance occurs in either split.

## Question 5 [50 points]: Classify the sentiment of the reviews using the combinations of various classifiers and vectorizers.
In this assignment, you should use the following list of models with default parameters:
```
models= {'knn': KNeighborsClassifier(), 'dt': DecisionTreeClassifier(), 'rf': RandomForestClassifier(), 'nb':MultinomialNB()}
```
and the following list of vectorizers:
```
vectorizers = {'bow_binary':CountVectorizer(ngram_range = (1,1), binary=True), \
               'bow_count': CountVectorizer(ngram_range = (1,1)), \
               'tfidf': TfidfVectorizer()}
```
The total number of combinations is 12. Compare the performance of the 12 model combinations in terms of `precision`, `recall`, and `f1-measure`. Visualize the performance as illustrated in the following figure:
![performance_comparison](https://i.imgur.com/Z4QKCmp.png)

In [ ]:
#Models
from sklearn.metrics import precision_score, recall_score, f1_score

model = {
    'knn': KNeighborsClassifier(),
    'dt': DecisionTreeClassifier(),
    'rf': RandomForestClassifier(),
    'nb': MultinomialNB()

}
#Ventorizers
vectorizers = {
    'bow_binary': CountVectorizer(ngram_range=(1,1), binary=True),
    'bow_count': CountVectorizer(ngram_range=(1,1)),
    'tfidf': TfidfVectorizer()
}
results = []

#loop through all combination
for model_name, model in model.items():
    for vec_name, vectorizer in vectorizers.items():
      # fit vectorizer in training data
      X_train_vec = vectorizer.fit_transform(review_train)
      X_test_vec = vectorizer.transform(review_test)

      #train model
      model.fit(X_train_vec, y_train)

      #predict
      y_pred = model.predict(X_test_vec)
      # Evaluate
      precision = precision_score(y_test, y_pred)
      recall = recall_score(y_test, y_pred)
      f1 = f1_score(y_test, y_pred)

      results.append({
            'combination': f"{model_name}_{vec_name}",
            'precision': precision,
            'recall': recall,
            'f1': f1
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

print(results_df)
# Evaluate
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

results.append({
    'combination': f"{model_name}_{vec_name}",
    'precision': precision,
    'recall': recall,
    'f1': f1
})

# Convert to DataFrame
results_df = pd.DataFrame(results)

print(results_df)

plt.figure(figsize=(12,6))

plt.plot(results_df['combination'], results_df['precision'], marker='o', label='precision')
plt.plot(results_df['combination'], results_df['recall'], marker='o', label='recall')
plt.plot(results_df['combination'], results_df['f1'], marker='o', label='f1-measure')

plt.xticks(rotation=45)
plt.ylabel("Performance Value")
plt.title("Classification Performance of Various Combinations of Classifiers and Vectorizers")
plt.legend()
plt.tight_layout()
plt.show()


## Question 6 [20 points]: Discuss the performance of the various model and vectorizer combinations. Which type of model gave a better performance? Do the vectorization methods affect the performance? Do you have any suggestions on changing the model parameters that could improve the performance? Why?

In [ ]:
# From the results, Naive Bayes (NB) and Random Forest (RF) clearly outperform KNN and Decision Tree across most metrics.
#Which model performed best?
#Overall, Naive Bayes provided the best performance, especially in terms of precision (highest precision = 0.8924 with TF-IDF).
# Naive Bayes works particularly well for text classification.

# Random Forest also performed strongly but slightly below Naive Bayes in F1-score.

# KNN performed the worst because: Distance-based methods struggle in high-dimensional sparse feature spaces.

# Do vectorization methods affect performance?

# Yes, vectorization clearly affects performance.
# For Naive Bayes, binary Bag-of-Words gave the best F1-score.
# For Random Forest, count vectorization slightly outperformed TF-IDF.

# Suggestions to Improve Performance

# Tune Model Parameters
# Random Forest Increase n_estimators , Adjust max_depth, Tune min_samples_split, Use max_features='sqrt'
# Decision Tree Limit max_depth , Increase min_samples_leaf


In [ ]:
#